# Generate verdicts and build report.json

The **G** in RAG. For each SRS requirement, an LLM (Gemini) reads the requirement, the NASA clauses
retrieved in `07_retrieve_relevant_clauses.ipynb`, and a few rules that always apply, then writes a
verdict. Lauren's models in `report_schema.py` check every verdict, and the verdicts are assembled
into `data/report.json` and checked with Mark's `validate_report.py`.

**Who decides what.** The LLM decides the status, the issue codes, the quotes, and which clauses
support each issue, citing clauses by ID. The code fills in the exact NASA section and URL from
`addressable_clauses.csv`, so a citation can't be made up.

**Setup.** Put `GOOGLE_API_KEY=...` in a `.env` file at the repo root (git ignores it). Answers are
cached in `data/llm_cache/`, so re-running only calls the API for requirements that haven't been
reviewed yet; if a run stops (e.g. Gemini is busy), just run it again. Delete a file there to review
that requirement again.

In [ ]:
import json, sys, time
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Literal
from urllib.parse import urldefrag

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "rag_retrievals.csv").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "contract"))
from report_schema import (DOCUMENT_TO_AUTHORITY, SOURCE_TO_DOCUMENT, EvidenceStrength, IssueCode,
                           QualityStatus, Report, RequirementReview, Severity, SoftwareRequirementStatus)
from validate_report import validate

load_dotenv(ROOT / ".env")

MODEL = "gemini-3.5-flash"
CACHE_DIR = ROOT / "data" / "llm_cache" / MODEL
REPORT_PATH = ROOT / "data" / "report.json"
MAX_ATTEMPTS = 3    # tries per requirement before falling back to UNDETERMINED
PAUSE_SECONDS = 4   # between API calls, to stay under the free tier's rate limit

In [ ]:
srs = pd.read_csv(ROOT / "data" / "srs_requirements.csv")
retrievals = pd.read_csv(ROOT / "data" / "rag_retrievals.csv")
clauses = pd.read_csv(ROOT / "data" / "addressable_clauses.csv").set_index("chunk_id")
srs_text = (ROOT / "data" / "Starhawk Mission Computer SRS.md").read_text(encoding="utf-8")
benchmark = pd.read_csv(ROOT / "data" / "benchmark" / "benchmark_inputs_for_teams_v1.csv")
calibration = pd.read_csv(ROOT / "data" / "benchmark" / "calibration_examples_for_teams_v1.csv")

print(len(srs), "requirements |", len(clauses), "clauses |", retrievals.req_id.nunique(), "requirements with retrievals")

## Clauses shown to the LLM

Each requirement gets its top 5 retrieved clauses plus three that always apply. Retrieval finds
clauses on the same *topic* as the requirement, so it never surfaces the general writing rules:
in 07, `swehb-1.b` ("specific, testable, and traceable") was in nobody's top 5.

| Always included | Why |
|---|---|
| `swehb-1.b-guidance` | Each requirement should be specific, testable, and traceable |
| `swehb-3.2.1-guidance` | Characteristics of functional requirements (clear, verifiable, behavioral, ...) |
| `npr-4.1.4-requirement` | SWE-184: safety constraints and controls belong in the requirements |

The whole SRS is included too (about 12 KB), so the LLM can use the open items, mode definitions,
and interfaces when judging one requirement.

In [ ]:
ALWAYS_INCLUDE = ["swehb-1.b-guidance", "swehb-3.2.1-guidance", "npr-4.1.4-requirement"]

def clauses_for(req_id):
    retrieved = retrievals.loc[retrievals.req_id == req_id].sort_values("rank").chunk_id.tolist()
    return list(dict.fromkeys(retrieved + ALWAYS_INCLUDE))

def clause_block(ids):
    return "\n\n".join(f"[{cid}] {clauses.at[cid, 'source']} §{clauses.at[cid, 'section']}\n{clauses.at[cid, 'text']}"
                       for cid in ids)

def locator(row):
    """'3.3 Targeting and Weapons' -> '§3.3, SFMC-REQ-015' (the format in Mark's sample)."""
    return f"§{row.section.split()[0].rstrip('.')}, {row.req_id}"

print(clauses_for("SFMC-REQ-015"))

## What the LLM fills in

A smaller version of Lauren's `RequirementReview`. The LLM doesn't type requirement IDs, raw text,
NASA sections, or URLs; the code adds those. It also writes a short `analysis` first, so it reasons
before it decides.

SWE links are limited to **SWE-184**. Of the six §4.1 provisions in Mark's contract, it's the only
one whose content (safety constraints and controls) can appear in a requirement's text; the others
are process rules that an SRS can't show.

In [ ]:
class Quote(BaseModel):
    locator: str = Field(description="Where the quote is from. For the requirement under review, use exactly the "
                                     "locator given in the prompt. For other SRS text, use its section, e.g. '§11, Open Item 10'.")
    excerpt: str = Field(description="Words copied exactly from the SRS, as short as possible.")

class DraftIssue(BaseModel):
    issue_code: IssueCode
    severity: Severity
    explanation: str
    evidence: List[Quote] = Field(min_length=1)
    clause_ids: List[str] = Field(min_length=1, description="IDs of the provided clauses that support this issue.")
    recommendation: str
    requires_stakeholder_input: bool

class DraftSweLink(BaseModel):
    swe_id: Literal["SWE-184"]
    strength: EvidenceStrength
    explanation: str
    evidence: List[Quote] = Field(min_length=1)

class DraftReview(BaseModel):
    analysis: str = Field(description="Brief reasoning before deciding: check the requirement against each issue type.")
    software_requirement_status: SoftwareRequirementStatus
    quality_issues: List[DraftIssue]
    swe_links: List[DraftSweLink]
    quality_status: QualityStatus
    summary: str = Field(description="One or two evidence-grounded sentences.")

## Draft → contract format

`to_review` turns a draft into Lauren's `RequirementReview`, which runs all of her checks (IDs, quotes
found in the SRS, citations among the 62 clauses, ACCEPTABLE means no issues). Any failure is sent
back to the LLM to fix.

In [ ]:
LINK_CLAUSE = {"SWE-184": "npr-4.1.4-requirement"}
STRENGTH_ORDER = ["DIRECT_CONTENT", "PARTIAL_CONTENT", "INDIRECT_CONTEXT"]

def nasa_ref(cid):
    c = clauses.loc[cid]
    document = SOURCE_TO_DOCUMENT[c.source]
    return {
        "document": document,
        "authority": DOCUMENT_TO_AUTHORITY[document],
        "section": f"§{c.section}",
        "swe_id": c.swe_id if isinstance(c.swe_id, str) else None,
        "url": urldefrag(c.source_url)[0],
    }

def to_review(row, draft, allowed):
    unknown = {cid for issue in draft.quality_issues for cid in issue.clause_ids} - set(allowed)
    if unknown:
        raise ValueError(f"clause_ids that were not provided: {sorted(unknown)}. Cite only the IDs in square brackets.")
    links = [{
        "swe_id": l.swe_id,
        "strength": l.strength,
        "explanation": l.explanation,
        "srs_evidence": [q.model_dump() for q in l.evidence],
        "nasa_reference": nasa_ref(LINK_CLAUSE[l.swe_id]),
    } for l in draft.swe_links]
    primary = min(links, key=lambda l: STRENGTH_ORDER.index(l["strength"]), default=None)
    return RequirementReview.model_validate({
        "requirement_id": row.req_id,
        "raw_text": row.text,
        "software_requirement_status": draft.software_requirement_status,
        "quality_status": draft.quality_status,
        "quality_issues": [{
            **issue.model_dump(exclude={"evidence", "clause_ids"}),
            "srs_evidence": [q.model_dump() for q in issue.evidence],
            "nasa_references": [nasa_ref(cid) for cid in dict.fromkeys(issue.clause_ids)],
        } for issue in draft.quality_issues],
        "primary_swe_id": primary and primary["swe_id"],
        "primary_swe_strength": primary and primary["strength"],
        "swe_links": links,
        "summary": draft.summary,
    })

def fallback(row):
    """Used when every attempt fails; not cached, so the next run tries again."""
    return RequirementReview.model_validate({
        "requirement_id": row.req_id, "raw_text": row.text,
        "software_requirement_status": "UNCERTAIN", "quality_status": "UNDETERMINED",
        "quality_issues": [], "primary_swe_id": None, "primary_swe_strength": None, "swe_links": [],
        "summary": f"Automated review failed validation after {MAX_ATTEMPTS} attempts; needs human review.",
    })

## Prompt

The rules come from Mark's output contract and the team's quality taxonomy. They describe each issue
type in general terms only: no benchmark requirement or answer appears in the prompt, so the
calibration comparison at the end stays a fair check.

In [ ]:
SYSTEM = """You review one requirement from the Starhawk Mission Computer SRS (SH-SRS-001, the SFMC).
Judge how well the requirement is written using the NASA software engineering handbook topic 5.09
(NASA-HDBK-2203), and link it to NPR 7150.2D SWE-184 only where its text justifies that.

software_requirement_status
- YES: it states behavior the SFMC software must provide (even if phrased around the pilot).
- NO: it is clearly not an obligation on the software.
- UNCERTAIN: it is unclear which system the obligation is allocated to.

quality_status
- ACCEPTABLE: no demonstrated writing defect. This is not a certificate of perfection; quality_issues must be empty.
- NEEDS_REVIEW: at least one specific defect you can show with an exact quote.
- UNDETERMINED: the SRS does not give enough to judge; explain what is missing.

Issue codes (use the most specific one; a requirement may have more than one issue)
- VAGUE_CRITERION: a degree word with no measurable bound or threshold (quickly, accurate, close, sufficient, noticeable).
- UNDEFINED_TERM: depends on a category the SRS never defines (which items count as "critical", "safe", "important"),
  including circular definitions where the condition is defined by the behavior it requires.
- UNSPECIFIED_RESPONSE: a trigger is given, but the required behavior is a placeholder ("appropriate action").
- ALLOCATION_UNCLEAR: the subject is not the SFMC software, references a system outside its documented interfaces,
  or leaves responsibility unassigned.
- UNVERIFIABLE_ABSOLUTE: a universal claim over unbounded time or inputs that no finite test can establish
  ("shall never fail"). "Never" or "always" attached to a clear, testable inhibit or capability is not this defect by itself.
- ESCAPE_CLAUSE: a qualifier that excuses non-compliance ("whenever possible", "as appropriate").
- SCOPE_AMBIGUITY: unclear which modes, conditions, or data the requirement applies to.
- SAFETY_INTERACTION: may interact with a safety function or inhibit in a way the SRS leaves unresolved (precedence
  unclear). Use this for a possible interaction, not a proven contradiction.
- OVERLAP_CLARIFICATION: overlaps another requirement and it is unclear which one governs.
- INCOMPLETE_TRACEABILITY, MISSING_QUALIFICATION_PROVISIONS, MISSING_ACCEPTANCE_CRITERIA, UNNUMBERED_REQUIREMENTS,
  INCOMPLETE_SAFETY_DEFINITIONS: document-level problems; use only if this requirement itself shows the problem.
- OTHER: a real defect none of the above covers; explain it.
Several obligations in one sentence is not a defect by itself.

Severity: CRITICAL if the defect leaves a safety function unverifiable or unimplementable; MAJOR if it makes the
requirement unverifiable or forces a design decision; MINOR otherwise.

Evidence and citations
- Every quote must be copied exactly from the SRS text provided. Keep quotes short.
- Cite NASA clauses only by the IDs shown in square brackets, choosing ones that actually support the issue.
- Handbook text is guidance. Never describe not following the Handbook as violating an NPR requirement.
- Recommendations must not invent numbers, mission parameters, or approval status. If stakeholders must decide
  something, say so and set requires_stakeholder_input to true.

SWE-184 link (NPR 7150.2D §4.1.4: safety constraints, controls, mitigations, and assumptions between hardware,
operator, and software belong in the software requirements)
- DIRECT_CONTENT: the requirement explicitly states a safety constraint or control.
- PARTIAL_CONTENT: it states one, but its criteria or scope are unresolved.
- INDIRECT_CONTEXT: related to safety, but not enough to say a safety control is captured.
- Leave swe_links empty when there is no safety content. Never link because words merely overlap. Most requirements
  have no link.
"""

def prompt_for(row, allowed):
    return f"""REQUIREMENT UNDER REVIEW
{row.req_id} ({row.title}) — locator for quotes from it: "{locator(row)}"
{row.text}

NASA CLAUSES (cite by the ID in square brackets)
{clause_block(allowed)}

FULL SRS
{srs_text}"""

## Calling the LLM

`init_chat_model` keeps the provider in one string, so switching to another LLM later means changing
`MODEL` above and this line, then installing that provider's package and key.

In [ ]:
def make_reviewer():
    from langchain.chat_models import init_chat_model
    llm = init_chat_model(f"google_genai:{MODEL}", max_retries=1)  # every retry counts against the free daily limit
    return llm.with_structured_output(DraftReview, method="json_schema")

reviewer = make_reviewer()

def call_llm(messages):
    """This waits and retries when Gemini is busy (503) or rate-limited (429)."""
    for wait in [30, 60, 120, None]:
        try:
            return reviewer.invoke(messages)
        except ValueError:
            raise  # a bad answer, handled by the caller
        except Exception as e:
            if wait is None or "PerDay" in str(e):  # daily quota: waiting won't help
                raise
            print(f"API unavailable ({type(e).__name__}), waiting {wait}s")
            time.sleep(wait)

def review_requirement(row):
    """Returns (review, attempts). attempts is 0 for a cached answer and None for a fallback."""
    cache = CACHE_DIR / f"{row.req_id}.json"
    allowed = clauses_for(row.req_id)
    if cache.exists():
        return to_review(row, DraftReview.model_validate_json(cache.read_text(encoding="utf-8")), allowed), 0

    messages = [("system", SYSTEM), ("human", prompt_for(row, allowed))]
    error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        draft = None
        try:
            draft = call_llm(messages)
            time.sleep(PAUSE_SECONDS)
            if draft is None:
                raise ValueError("no structured answer was returned")
            review = to_review(row, draft, allowed)
        except ValueError as e:  # includes Pydantic's ValidationError and LangChain's parsing errors
            error = e
            answer = draft.model_dump_json() if isinstance(draft, DraftReview) else "(unreadable answer)"
            messages += [("ai", answer), ("human", f"That answer failed these checks:\n{e}\n\nFix them and answer again.")]
            continue
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text(draft.model_dump_json(indent=2), encoding="utf-8")
        return review, attempt
    print(f"{row.req_id}: falling back to UNDETERMINED: {error}")
    return fallback(row), None

## Try a few first

One good requirement, one with a missing response, and two safety cases. Check these before running
all 54.

In [ ]:
TRY_FIRST = ["SFMC-REQ-002", "SFMC-REQ-010", "SFMC-REQ-015", "SFMC-REQ-035"]
attempts = {}

for _, row in srs[srs.req_id.isin(TRY_FIRST)].iterrows():
    review, attempts[row.req_id] = review_requirement(row)
    codes = ", ".join(i.issue_code for i in review.quality_issues) or "-"
    print(f"{review.requirement_id}  {review.quality_status:13} {codes:40} SWE: {review.primary_swe_id or '-'}  (attempts: {attempts[row.req_id]})")
    print(f"    {review.summary}\n")

## Review all 54

In [ ]:
reviews = []
for _, row in srs.iterrows():
    review, a = review_requirement(row)
    attempts.setdefault(row.req_id, a)  # keep the count from the try-first cell
    reviews.append(review)

fallbacks = [rid for rid, a in attempts.items() if a is None]
print(len(reviews), "reviewed |", sum(a == 0 for a in attempts.values()), "from cache |",
      sum(bool(a) and a > 1 for a in attempts.values()), "needed a retry |", len(fallbacks), "fell back")
pd.Series([r.quality_status for r in reviews]).value_counts()

## Assemble and validate report.json

Only `requirement_reviews` is generated so far, so the run is `DEMO` scope. A `FULL_SRS` run also
needs the six SWE summaries and the document findings.

In [ ]:
limitations = [
    "Requirement reviews only; relationships, document findings, and SWE coverage are not generated yet.",
    "No NASA project compliance is determined from the SRS alone.",
    "Verdicts are LLM-generated and have not been checked by a human reviewer.",
]
if fallbacks:
    limitations.append(f"{len(fallbacks)} requirement(s) failed automated review and are marked UNDETERMINED: {', '.join(fallbacks)}.")

report = Report.model_validate({
    "schema_version": "1.0.0",
    "run": {
        "input_document_id": "SH-SRS-001",
        "input_document_version": "1.0",
        "input_source": "https://github.com/Break-Through-Tech/Aerospace-1B-compliance-copilot/blob/main/data/Starhawk%20Mission%20Computer%20SRS.md",
        "evaluation_scope": "DEMO",
        "npr_reference_version": "NPR-7150.2D",
        "handbook_reference": "NASA-HDBK-2203-5.09",
        "method": (f"RAG: top-5 clauses retrieved from Chroma (all-MiniLM-L6-v2) plus {len(ALWAYS_INCLUDE)} always-included "
                   f"clauses; verdicts by {MODEL} with structured output; citations filled from addressable_clauses.csv; "
                   "validated with report_schema.py"),
        "generated_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "limitations": limitations,
    },
    "requirement_reviews": [r.model_dump() for r in reviews],
    "relationships": [],
    "document_findings": [],
    "swe_coverage": [],
})
report.to_json(REPORT_PATH)
print("validate_report.py counts (reviews, relationships, document findings, SWEs):", validate(REPORT_PATH))

## Compare with the calibration examples

Mark's 7 requirement-level calibration answers. A match on status and on the expected issue code is
the target; differences are worth reading, since some calls are judgment calls.

In [ ]:
by_id = {r.requirement_id: r for r in reviews}
cal = calibration[calibration.scope == "REQUIREMENT"][["target_ids", "expected_outcome", "issue_code"]].copy()
cal["predicted"] = cal.target_ids.map(lambda i: by_id[i].quality_status)
cal["predicted_codes"] = cal.target_ids.map(lambda i: ", ".join(q.issue_code for q in by_id[i].quality_issues) or "NONE")
cal["status_ok"] = cal.predicted == cal.expected_outcome
cal["code_ok"] = [code in codes.split(", ") for code, codes in zip(cal.issue_code, cal.predicted_codes)]
print(f"status {cal.status_ok.sum()}/{len(cal)} | issue code {cal.code_ok.sum()}/{len(cal)}")
cal

## Checks

In [ ]:
bench_reqs = set(benchmark.loc[benchmark.scope == "REQUIREMENT", "target_ids"])
assert len(reviews) == len(srs) == 54, "every requirement gets exactly one review"
assert bench_reqs <= set(by_id), f"benchmark requirements missing: {sorted(bench_reqs - set(by_id))}"
assert json.loads(REPORT_PATH.read_text(encoding="utf-8"))["requirement_reviews"], "report.json was written"
print(f"all {len(bench_reqs)} benchmark requirements reviewed; report.json passes validate_report.py")